In [11]:
import sys
!{sys.executable} -m pip install rioxarray

  Using cached rioxarray-0.15.1-py3-none-any.whl (53 kB)
  Using cached xarray-2024.2.0-py3-none-any.whl (1.1 MB)


In [12]:
import rioxarray

In [8]:
from pysal.lib import weights

In [14]:
# Open GeoTIFF file and read into `xarray.DataArray`
pop = rioxarray.open_rasterio("G:\My Drive\INVESTIGACION\POSDOC\Data\Raster/DEM_30m_recortado.tif")

In [15]:
w_surface_sp = weights.Queen.from_xarray(pop)

In [17]:
w_surface_all = weights.WSP2W(weights.WSP(w_surface_sp.sparse.astype(float),id_order=w_surface_sp.index.tolist()))
w_surface_all.index = w_surface_sp.index

MemoryError: Unable to allocate 6.95 GiB for an array with shape (933255784,) and data type float64

In [ ]:
# Convert `DataArray` to a `pandas.Series`
pop_values = pop.to_series()
# Subset to keep only values that aren't missing
pop_values = pop_values[pop_values != pop.rio.nodata]

In [ ]:
pop.rio.nodata

In [ ]:
w_surface = weights.w_subset(w_surface_all, pop_values.index)
w_surface.index = pop_values.index

In [ ]:
# NOTE: this may take a bit longer to run depending on hardware
pop_lisa = esda.moran.Moran_Local(
    pop_values.astype(float), w_surface, n_jobs=-1
)

In [ ]:
from libpysal.weights import raster


In [ ]:
sig_pop = pandas.Series(
    pop_lisa.q
    * (
        pop_lisa.p_sim < 0.01
    ),  # Quadrant of significant at 1% (0 otherwise)
    index=pop_values.index,  # Index from the Series and aligned with `w_surface`
)

In [ ]:
lisa_da.to_series().unique()

In [ ]:
from matplotlib.colors import ListedColormap

In [ ]:
# LISA colors
lc = {
    "ns": "lightgrey",  # Values of 0
    "HH": "#d7191c",  # Values of 1
    "LH": "#abd9e9",  # Values of 2
    "LL": "#2c7bb6",  # Values of 3
    "HL": "#fdae61",  # Values of 4
}

In [ ]:
lisa_cmap = ListedColormap(
    [lc["ns"], lc["HH"], lc["LH"], lc["LL"], lc["HL"]]
)
lisa_cmap

In [ ]:
# Set up figure and axis
f, axs = plt.subplots(1, 2, figsize=(12, 6))
# Subplot 1 #
# Select pixels that do not have the `nodata` value
# (ie. they are not missing data)
pop.where(
    pop
    != pop.rio.nodata
    # Plot surface with a horizontal colorbar
).plot(
    ax=axs[0],
    add_colorbar=False,  # , cbar_kwargs={"orientation": "horizontal"}
)
# Subplot 2 #
# Select pixels with no missing data and rescale to [0, 1] by
# dividing by 4 (maximum value in `lisa_da`)
(
    lisa_da.where(lisa_da != -200)
    / 4
    # Plot surface without a colorbar
).plot(cmap=lisa_cmap, ax=axs[1], add_colorbar=False)
# Aesthetics #
# Subplot titles
titles = ["Population by pixel", "Population clusters"]
# Apply the following to each of the two subplots
for i in range(2):
    # Keep proportion of axes
    axs[i].axis("equal")
    # Remove axis
    axs[i].set_axis_off()
    # Add title
    axs[i].set_title(titles[i])
    # Add basemap
    contextily.add_basemap(axs[i], crs=lisa_da.rio.crs)